# RV3D — Eikonal Weight Ablation

Sweeps the eikonal loss weight over six values and three random seeds
(18 training runs) and selects the value used in
`02_train_and_evaluate.ipynb`.

**This is the benchmark where the eikonal weight actually matters.**
Translation, rotation and Zalesak are all rigid motions: the exact solution
stays an exact signed distance function, so the eikonal term is free
supervision. Under the reversed vortex the interface stretches into a folded
sheet, the advected field stops being an SDF, and enforcing |grad phi| = 1
competes directly with the transport residual.

The 2D counterpart of this test used `w_eik = 0`, so a value at or near zero is
the expected outcome. Confirming that in 3D — or contradicting it — is the
point of the sweep.

**Note on T.** T = 2 gives milder deformation than the T = 3 of Enright et al.
The interface therefore departs less from a signed distance function, so the
eikonal conflict is weaker than it would be at larger T. If the sweep shows
little sensitivity, that is partly a consequence of T and should be reported as
such rather than as evidence that the term is harmless under deformation.

**Reference solution.** There is no closed form. `reference_fields` back-traces
characteristics with RK4 and caches the result; it does not depend on the
network, so all 18 runs reuse one computation. The self-consistency check at
t = T (where the reference must reproduce the initial field exactly) is printed
when it is computed.

**Outputs:** `RV3D_eikonal_ablation.csv` with per-snapshot and averaged metrics
for every run, plus a summary table with seed spread. Rows are appended after
each run and completed runs are skipped on restart.

In [1]:
# --- setup -------------------------------------------------------------
import sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/RV3D'
except ImportError:
    PROJECT_DIR = os.getcwd()

sys.path.insert(0, PROJECT_DIR)
os.makedirs(PROJECT_DIR, exist_ok=True)

import csv, itertools, time
import numpy as np
import torch
import rv3d_core as core

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

Mounted at /content/drive
device: cuda


In [2]:
# --- configuration -----------------------------------------------------
W_EIK_GRID   = [0.0, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
SEEDS        = [42, 43, 44]

W_PDE, W_IC  = 1.0, 10.0
N_F, N_I     = 10000, 5000
LAYERS, NEURONS = 8, 256
T_FINAL      = 2.0
ADAM_EPOCHS  = 20000        # same budget as the reported run, so the ranking
LBFGS_EPOCHS = 2000         # transfers without extrapolation
GRID_N       = 201
N_SNAPSHOTS  = 6
RK4_STEPS    = 200

M_CHUNKS, EPS_CAUSAL = 10, 1.0
RAD_FREQ, RAD_CANDIDATES, RAD_K, RAD_C = 1000, 20000, 1.0, 1.0
RAR_FREQ, RAR_CANDIDATES, RAR_ADD      = 1800, 50000, 500

EST_MIN_PER_RUN = 15        # update from your first measured run
CSV_PATH  = f'{PROJECT_DIR}/RV3D_eikonal_ablation.csv'
REF_CACHE = f'{PROJECT_DIR}/RV3D_reference_n{GRID_N}_T{T_FINAL:g}.npz'

times, tlabels = core.snapshot_times(T_FINAL, N_SNAPSHOTS)
grid = core.Grid(n=GRID_N)

# computed once and reused by every run in the sweep
PHI_E = core.reference_fields(grid, times, T_FINAL, device,
                              nsteps=RK4_STEPS, cache_path=REF_CACHE)

METRICS = ['rel_L2', 'vol_MAPE', 'eik_dev', 'l2_abs', 'vol_abs', 'vol_floor']
FIELDS = ['w_eik', 'seed']
for m in METRICS:
    FIELDS += [f'{m}_t{i}' for i in range(N_SNAPSHOTS)] + [f'avg_{m}']
FIELDS += ['loss_adam', 'loss_final', 'lbfgs_gain', 'n_points', 'minutes']

print(f'{len(W_EIK_GRID) * len(SEEDS)} runs, {len(FIELDS)} columns per row')

loaded reference fields from RV3D_reference_n201_T2.npz
18 runs, 49 columns per row


In [3]:
# --- one run -----------------------------------------------------------
def run_one(w_eik, seed):
    net = core.build_model(device, seed=seed, layers=LAYERS, neurons=NEURONS)
    X_f, X_i = core.sample_points(N_F, N_I, device, seed=seed)
    sampler = core.AdaptiveSampler(net, X_f, T_FINAL, device,
                                   rad_freq=RAD_FREQ, rad_candidates=RAD_CANDIDATES,
                                   rad_k=RAD_K, rad_c=RAD_C,
                                   rar_freq=RAR_FREQ, rar_candidates=RAR_CANDIDATES,
                                   rar_add=RAR_ADD)
    loss_fn = core.make_loss(net, sampler, X_i, T_FINAL, W_PDE, W_IC, w_eik,
                             m_chunks=M_CHUNKS, eps_causal=EPS_CAUSAL)

    tr = core.train(net, loss_fn, sampler=sampler, adam_epochs=ADAM_EPOCHS,
                    lbfgs_epochs=LBFGS_EPOCHS, verbose=False)
    res = core.evaluate(net, grid, times, T_FINAL, device, PHI_E, cache=False)

    row = {'w_eik': w_eik, 'seed': seed}
    for m in METRICS:
        for i in range(N_SNAPSHOTS):
            row[f'{m}_t{i}'] = f'{res[m][i]:.6e}'
        row[f'avg_{m}'] = f"{res['avg_' + m]:.6e}"
    row.update({'loss_adam': f"{tr['loss_adam']:.6e}",
                'loss_final': f"{tr['loss_final']:.6e}",
                'lbfgs_gain': f"{tr['lbfgs_gain']:.6e}",
                'n_points': tr['n_points'],
                'minutes': f"{tr['minutes']:.2f}"})

    del net, X_f, X_i, sampler
    if device == 'cuda':
        torch.cuda.empty_cache()
    return row

In [4]:
# --- sweep (resumable) -------------------------------------------------
done = set()
if os.path.exists(CSV_PATH):
    with open(CSV_PATH) as f:
        for r in csv.DictReader(f):
            done.add((float(r['w_eik']), int(r['seed'])))
    print(f'{len(done)} run(s) already complete — skipping those')

todo = [p for p in itertools.product(W_EIK_GRID, SEEDS) if p not in done]
print(f'{len(todo)} run(s) to go, ~{len(todo) * EST_MIN_PER_RUN / 60:.1f} h\n', flush=True)

for k, (wv, sd) in enumerate(todo, 1):
    print(f'[{k}/{len(todo)}] w_eik={wv:g} seed={sd} ...', flush=True)
    row = run_one(wv, sd)
    new = not os.path.exists(CSV_PATH) or os.path.getsize(CSV_PATH) == 0
    with open(CSV_PATH, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)
    print(f"    relL2={float(row['avg_rel_L2']):.4f}%  "
          f"MAPE={float(row['avg_vol_MAPE']):.4f}%  "
          f"eik={float(row['avg_eik_dev']):.3e}  "
          f"lbfgs_gain={float(row['lbfgs_gain']):.2e}  "
          f"({float(row['minutes']):.1f} min)", flush=True)

18 run(s) already complete — skipping those
0 run(s) to go, ~0.0 h



## Summary and selection

Decision rule, fixed before the sweep:

1. **Primary** — lowest average relative L₂
2. **Tiebreak 1** — lowest average volume MAPE
3. **Tiebreak 2** — lowest eikonal deviation
4. **Tiebreak 3** — the smaller weight

Two weights count as *tied* when the gap between their means is smaller than
the larger of their two seed standard deviations. Eikonal deviation is only a
second-order tiebreak on purpose: it is essentially the quantity the swept term
minimises, so ranking by it would recover the sweep by construction.

In [5]:
# --- summary -----------------------------------------------------------
rows = list(csv.DictReader(open(CSV_PATH)))
agg = {}
for r in rows:
    agg.setdefault(float(r['w_eik']), []).append(r)

stats = {}
print(f"{'w_eik':>8} | {'rel L2 (%)':>19} | {'vol MAPE (%)':>19} | "
      f"{'eik dev':>11} | {'n':>2}")
print('-' * 74)
for wv in sorted(agg):
    g = agg[wv]
    m = {k: np.mean([float(r['avg_' + k]) for r in g]) for k in
         ('rel_L2', 'vol_MAPE', 'eik_dev')}
    s = {k: np.std([float(r['avg_' + k]) for r in g]) for k in
         ('rel_L2', 'vol_MAPE', 'eik_dev')}
    stats[wv] = (m, s, len(g))
    print(f"{wv:>8g} | {m['rel_L2']:>9.4f} ± {s['rel_L2']:<7.4f} | "
          f"{m['vol_MAPE']:>9.4f} ± {s['vol_MAPE']:<7.4f} | "
          f"{m['eik_dev']:>11.3e} | {len(g):>2}")

def tied(a, b, key):
    (ma, sa, _), (mb, sb, _) = stats[a], stats[b]
    return abs(ma[key] - mb[key]) < max(sa[key], sb[key])

best = min(stats, key=lambda w: stats[w][0]['rel_L2'])
for cand in sorted(stats, key=lambda w: stats[w][0]['rel_L2'])[1:]:
    if tied(best, cand, 'rel_L2'):
        if not tied(best, cand, 'vol_MAPE'):
            best = min(best, cand, key=lambda w: stats[w][0]['vol_MAPE'])
        elif not tied(best, cand, 'eik_dev'):
            best = min(best, cand, key=lambda w: stats[w][0]['eik_dev'])
        else:
            best = min(best, cand)

ties = sorted(w for w in stats if w != best and tied(best, w, 'rel_L2'))
print('-' * 74)
print(f'selected w_eik = {best:g}')
if ties:
    print('statistically tied on rel L2 with: ' + ', '.join(f'{w:g}' for w in ties))
    print('-> report insensitivity across this range rather than a single value')
print(f"mean L-BFGS gain over all runs: "
      f"{np.mean([float(r['lbfgs_gain']) for r in rows]):.3e}")

   w_eik |          rel L2 (%) |        vol MAPE (%) |     eik dev |  n
--------------------------------------------------------------------------
       0 |    0.5776 ± 0.0196  |    2.7321 ± 0.2539  |   6.115e-01 |  3
   1e-05 |    0.5879 ± 0.0264  |    1.1728 ± 0.6391  |   6.066e-01 |  3
  0.0001 |    4.8764 ± 1.5017  |   65.0133 ± 17.9320 |   4.887e-01 |  3
   0.001 |   10.5783 ± 0.8609  |   82.5859 ± 2.6150  |   2.055e-01 |  3
    0.01 |   11.1760 ± 0.5285  |   32.0027 ± 9.7102  |   1.764e-01 |  3
     0.1 |   15.1242 ± 1.6760  |   58.0118 ± 13.1353 |   2.264e-01 |  3
--------------------------------------------------------------------------
selected w_eik = 1e-05
statistically tied on rel L2 with: 0
-> report insensitivity across this range rather than a single value
mean L-BFGS gain over all runs: 5.108e-03


## Error growth per snapshot

Per-snapshot averages across seeds, for the time-evolution table/figure.

In [6]:
# --- per-snapshot table ------------------------------------------------
METRIC = 'rel_L2'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

rel_L2 by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    0.1362 |    0.4323 |    0.6743 |    0.7619 |    0.7407 |    0.7200
   1e-05 |    0.1525 |    0.4542 |    0.6889 |    0.7682 |    0.7466 |    0.7170
  0.0001 |    0.5612 |    4.0106 |    5.5654 |    6.4475 |    6.3732 |    6.3003
   0.001 |    0.8537 |   10.2661 |   12.8241 |   13.2706 |   13.2404 |   13.0146
    0.01 |    1.0801 |   10.7699 |   13.0453 |   13.9777 |   14.0664 |   14.1169
     0.1 |    1.3731 |   14.1474 |   17.4158 |   18.9713 |   19.4717 |   19.3658


In [8]:
# --- per-snapshot table ------------------------------------------------
METRIC = 'vol_MAPE'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

vol_MAPE by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    0.5579 |    1.5506 |    1.6841 |    3.0039 |    4.3122 |    5.2839
   1e-05 |    0.2850 |    0.9313 |    0.7731 |    1.0728 |    1.8135 |    2.1609
  0.0001 |    3.5388 |   52.1336 |   72.7471 |   83.9436 |   88.2564 |   89.4604
   0.001 |    4.4449 |   96.3722 |   99.6186 |   99.6042 |   97.7969 |   97.6787
    0.01 |    4.7154 |   38.0769 |   38.1321 |   37.5906 |   34.7509 |   38.7504
     0.1 |    6.8317 |   67.6531 |   68.4745 |   62.3665 |   70.7805 |   71.9645


In [9]:
# --- per-snapshot table ------------------------------------------------
METRIC = 'eik_dev'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

eik_dev by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    0.0219 |    0.5474 |    1.2595 |    1.2563 |    0.5252 |    0.0589
   1e-05 |    0.0227 |    0.5526 |    1.2475 |    1.2285 |    0.5312 |    0.0569
  0.0001 |    0.0607 |    0.4769 |    0.7683 |    0.8298 |    0.4609 |    0.3357
   0.001 |    0.1367 |    0.2622 |    0.2234 |    0.2730 |    0.1451 |    0.1925
    0.01 |    0.1046 |    0.1417 |    0.2360 |    0.2594 |    0.1483 |    0.1686
     0.1 |    0.2062 |    0.0957 |    0.4571 |    0.3129 |    0.1088 |    0.1776


In [10]:
# --- per-snapshot table ------------------------------------------------
METRIC = 'l2_abs'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

l2_abs by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    0.0006 |    0.0018 |    0.0029 |    0.0033 |    0.0032 |    0.0031
   1e-05 |    0.0007 |    0.0019 |    0.0029 |    0.0033 |    0.0032 |    0.0031
  0.0001 |    0.0024 |    0.0171 |    0.0237 |    0.0275 |    0.0272 |    0.0269
   0.001 |    0.0036 |    0.0438 |    0.0547 |    0.0566 |    0.0565 |    0.0555
    0.01 |    0.0046 |    0.0459 |    0.0557 |    0.0596 |    0.0600 |    0.0602
     0.1 |    0.0059 |    0.0604 |    0.0743 |    0.0809 |    0.0831 |    0.0826
